# Kev on MiniCPM5-2B-Base

Trains Kev decision models on `openbmb/MiniCPM5-2B-Base`, picks the best run and compares it with the released Kev-0.8B and Kev-4B.
Everything runs through `scripts/h200_minicpm5.sh`, **detached**: closing this notebook, the browser or SSH does not stop training.

**Run the cells top to bottom.** 1 configures, 2 fetches the code, 3 launches, 4 shows live progress, 5 shows results.

| Hardware | Runs at once | All 4 runs + evaluation (estimate) |
|---|---|---|
| 32 GB MIG slice | 1 | ~10-12 h |
| 1x H100 / RTX Pro 6000 | 2 | ~2-3 h / ~3 h |
| 1x H200 / B200 | 3 | ~1.5-2 h / ~1-1.5 h |
| 4x H100 / H200 / B200 | 4 (one per GPU) | ~1 h / ~45-60 min / ~35-45 min |

Each run is the Kev release recipe (decision-v7 training set, 2 epochs, batch 8, bf16, LoRA 16) and is also scored on transfer-v4 (sources it never trained on). The locked test set is never read.

## 1. Settings

In [ ]:
import os

REPO = "https://github.com/AyushChauhan9389/kev"   # private: cell 2 asks for a GitHub token if it has to clone
WORKDIR = os.path.expanduser("~/kev")             # where the code lives (ignored if this notebook is already inside the repo)

# minicpm5-2b-a: lr 1e-4, with and without training the delimiter embeddings (2 runs)
# minicpm5-2b-b: lr 5e-5, and lr 1e-4 with a second seed (2 runs)
LANES = "minicpm5-2b-a minicpm5-2b-b"   # "minicpm5-2b-a" alone = 2 runs, half the time
GPUS = "all"                            # or e.g. "0,1" to use only those GPUs
SLOTS = "auto"                          # runs per GPU; auto = by GPU memory (1 below 80 GB, 2 below 120 GB, else 3)
BASELINES = "jaredpalmer/kev-0.8b jaredpalmer/kev-4b"   # released models to compare the winner with ("" = none)

## 2. Get the code and check the GPUs

In [ ]:
import getpass, os, shutil, subprocess
from pathlib import Path

def sh(cmd, **kw):
    """Run a shell command, print its output, raise if it fails."""
    print(subprocess.run(cmd, shell=True, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw).stdout)

# inside the repo already (opened from a clone)? use it; else clone or update WORKDIR
here = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/h200_minicpm5.sh").exists()), None)
KEV = here or Path(WORKDIR)
if not (KEV / "scripts/h200_minicpm5.sh").exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token with repo read access (not stored): ")
    authed = REPO.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", authed, str(KEV)], check=True)
    subprocess.run(["git", "-C", str(KEV), "remote", "set-url", "origin", REPO], check=True)   # keep the token out of .git/config
else:
    r = subprocess.run(["git", "-C", str(KEV), "pull", "--ff-only"], text=True, capture_output=True)
    print(r.stdout or r.stderr)   # a private repo may refuse an unauthenticated pull; the checkout you have still works
os.chdir(KEV)
print("code:", KEV)

# uv drives the environment; install it for this user if missing
if not shutil.which("uv"):
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
sh("uv --version")
sh("nvidia-smi -L")

## 3. Launch

Returns within a second; the run continues in the background. First it installs packages (falls back to pip if uv cannot reach PyPI, as on some Kubernetes pods), runs the unit tests, and does a ~2 minute MiniCPM smoke training plus parity check. The real runs start after that.

Running it again while a run is active is refused. After a crash it resumes: finished runs are skipped, broken ones restarted.

In [ ]:
env = {**os.environ, "LANES": LANES, "GPUS": GPUS, "SLOTS": SLOTS, "BASELINES": BASELINES}
setup_done = Path(".venv/bin/python").exists() and Path("runs/h200-smoke/head.pt").exists()
env["PHASE"] = "resume" if setup_done else "all"   # resume = skip setup, train what is left, then evaluate
print(f"PHASE={env['PHASE']}")
print(subprocess.run(["scripts/h200_minicpm5.sh"], env=env, text=True, capture_output=True).stdout)

## 4. Progress

Re-run this cell any time. `watch()` refreshes every minute until the run ends (interrupt the cell to stop watching; training keeps going).

In [ ]:
import re, time
from IPython.display import clear_output

LOGS = Path("runs/h200-logs")
STEP = re.compile(r"ep(\d+) step (\d+)/(\d+) .*? ([\d.]+)s/rec")

def status():
    env = {**os.environ, "PHASE": "status"}
    out = subprocess.run(["scripts/h200_minicpm5.sh"], env=env, text=True, capture_output=True).stdout
    running = out.startswith("running")
    print("RUNNING" if running else "NOT RUNNING", "|", time.strftime("%H:%M:%S"))
    run_log = (LOGS / "run.log").read_text(errors="replace").splitlines() if (LOGS / "run.log").exists() else []
    print("\nlast lines of run.log:")
    for line in run_log[-8:]: print("  ", line[:160])
    print("\nruns:")
    for log in sorted(p for p in LOGS.glob("*.log") if p.stem not in ("run", "queue")):
        text = log.read_text(errors="replace")
        ledger = Path("runs", log.stem, "results.jsonl")   # kev.experiment writes it when a study ends
        steps = STEP.findall(text)
        if ledger.exists():
            state = "FAILED (see log)" if '"failed"' in ledger.read_text() else "done"
        elif "Traceback" in text:
            state = "FAILED (see log)"
        elif steps:
            ep, n, total, spr = steps[-1]; n, total, spr = int(n), int(total), float(spr)
            eta = (total - n) * 8 * spr / 60
            state = f"training {n}/{total} ({100 * n / total:.0f}%), {spr:.2f} s/record, ~{eta:.0f} min left" if n < total else "evaluating"
        else:
            state = "starting (loading model)"
        print(f"   {log.stem:22} {state}")
    return running

def watch(every=60):
    while True:
        clear_output(wait=True)
        if not status(): break
        time.sleep(every)

status()

In [ ]:
watch()   # live view; interrupt the cell (■) to stop watching, training continues

## 5. Results

In [ ]:
import json

def load(p): return json.loads(Path(p).read_text(encoding="utf-8"))

rows = []
for lane in LANES.split():
    studies = [Path("runs", lane), *[p for p in Path("runs").glob(f"{lane}-t*") if ".incomplete" not in p.name]]
    for result in sorted(r for s in studies for r in s.glob("*/result.json")):
        r = load(result); cfg = r["provenance"]["config"]
        rows.append((r["transfer"]["clean"]["acc"], r["clean"]["acc"], r["transfer"]["clean"]["brier"], cfg["lr"],
                     cfg.get("special_embeddings", 0), cfg["seed"], str(result.parent)))
print("Runs, best first (transfer-v4 = sources never trained on; decision-v7 = held-out rows of the training sources)")
print(f"{'transfer-v4 acc':>16} {'decision-v7 acc':>16} {'transfer brier':>15} {'lr':>8} {'emb':>4} {'seed':>5}  path")
for t, d, b, lr, se, seed, path in sorted(rows, reverse=True):
    print(f"{t:16.3f} {d:16.3f} {b:15.3f} {lr:8.0e} {se:4} {seed:5}  {path}")

ev = Path("runs/h200-eval")
if not ev.exists():
    print("\nFinal evaluation not written yet (it runs after all training).")
else:
    for suite in ("transfer-v4", "transfer-v9"):
        print(f"\n{suite} development  (acc higher is better; brier / ece lower)")
        for name in ["winner", *[b.split("/")[1] for b in BASELINES.split()]]:
            rep = ev / f"{name}-{suite}" / "report.json"
            if rep.exists():
                c = load(rep)["clean"]
                print(f"   {name:12} acc {c['acc']:.3f}   brier {c['brier']:.3f}   ece {c['ece']:.3f}   n={c['n']}")
        for b in BASELINES.split():
            cmp = ev / f"winner-vs-{b.split('/')[1]}-{suite}.json"
            if cmp.exists():
                p = load(cmp)["paired"]["acc"]
                d = next(v for k, v in p.items() if k.endswith("_delta")); lo, hi = p["ci95"]
                verdict = "better" if lo > 0 else "worse" if hi < 0 else "no clear difference"
                print(f"   winner vs {b.split('/')[1]}: accuracy {d:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}) -> {verdict}")

## Stop

Stops the run and every training job it started. Finished runs are kept; launching again (cell 3) resumes.

In [ ]:
print(subprocess.run(["scripts/h200_minicpm5.sh"], env={**os.environ, "PHASE": "stop"}, text=True, capture_output=True).stdout)